# Example Segmentation Methods for Li-ion Cylindrical Cell XCT
This notebook showcases **two lightweight segmentation approaches** you can use on Li-ion cylindrical cell XCT data:

**A. Mini U-Net (overhangs):** a compact deep-learning model trained on angle-sliced images to segment electrode **overhangs**.

**B. Classic CV (electrode winding):** a transparent thresholding + morphology pipeline that segments the **winding** quickly without a model.

**What's included?:**
- Mini U-net model, alongside data processing, augmentation, and training scripts (available in Zenodo repo alongside this notebook).
- Learn to fetch the pre-trained mini U-Net from Zenodo and run a fast slice-level inference.
- Visualize the segmentation overlay with a random label colormap.
- Compare this ML workflow to a classic CV approach for segmenting the electrode winding, ideal for making training data and quick analysis.


## 0) Setup & imports
Add the project root to `sys.path` for utility imports. 

In [ ]:
import os, sys
CWD = os.getcwd()
if os.path.basename(CWD) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(CWD, ".."))
else:
    PROJECT_ROOT = CWD  # fallback if already at root
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "utils"))
print(f"Correct Working Directory: {str(os.path.basename(os.getcwd()))=='battery_xct_workflows'}")

Pull in the scientific Python stack and plotting helpers.

In [ ]:
import tifffile as tiff
import numpy as np
import keras
import pooch
import cv2
from scipy.ndimage import label
from skimage.morphology import erosion, dilation, star, remove_small_objects
from utils.plotting_utils import overlay_multilabel
import matplotlib.pyplot as plt

## 1) U-Net overhang segmentation (mini model)

We load a compact U-Net (ResNet-50 encoder) trained on angle-sliced images and run a **single-slice** overhang segmentation (see notebook #1 for angular slicing method)

**Open Source Model**
- Open source model trained on a 'mini' dataset is made available alongside this notebook at https://zenodo.org/records/17543023
- We have also made available the training set and scripts for data wrangling, augmentation and model training.

**Model I/O**
- Model input is resized to `(IMG_HEIGHT, IMG_WIDTH)`.
- Prediction is resized back to the original image size.
- A simple threshold converts the probability map to a binary mask, then `label()` seperates individual overhangs.

In [ ]:
# Get model backbone
BACKBONE = 'resnet50'
# Model Input dimensions
IMG_CHANNELS = 3
IMG_HEIGHT = 256
IMG_WIDTH = 800

# Download the model from the Zenodo repository 
url = "https://zenodo.org/records/17543023/files/unet_resnet_overhangs_model_mini2.keras?download=1"
fname = pooch.retrieve(
    url=url,
    fname="unet_resnet_overhangs_model_mini2.keras",
    known_hash="md5:7c287f71a9b4bd7cfc76fad6e53457b5",          
    progressbar=True,
)

# Load model
model = keras.models.load_model(fname, compile=False)

### Load a sample slice
We start with a single 2D angle slice. This allows for warp free sampling of the overhang region as discussed in notebook #1. 


In [ ]:
im = tiff.imread('data/overhang.tif')
og_shape = im.shape # For later resizing output to original dimensions 
plt.imshow(im, cmap='gray')
plt.show()

### Define a minimal segmentation function and visualize
This function handles **preprocessing**, **inference**, and **postprocessing**:
1. Convert grayscale → RGB.
2. Normalize intensities to `[0, 1]`.
3. Resize to network input size, predict probability map.
4. Segment via thresholding of the probability map
5. Resize back to original, threshold to binary, then label connected components.
6. Visualize overhangs rendered in a random colormap overlayed on original greyscale slice

In [ ]:
def segment(img, model = model, p_thresh = 0.9):
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)  
    img = (img/np.amax(img)).astype(np.float32)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img, verbose = 0)
    pred = cv2.resize(pred[0,:,:,0], (og_shape[1], og_shape[0]))
    img_seg = np.where(pred>p_thresh, 1, 0)
    img_seg, _num_obj = label(img_seg)
    return np.array(img_seg)

segmented = segment(im)
overlay_multilabel(im, segmented, title = 'U-net Overhang Segmentation')

**Remarks**
- This mini model is designed to be an easy to use demonstration. For production, train on a larger, more diverse dataset and validate across cells.

## 2) Classic CV segmentation (electrode winding)
Now we apply a thresholding + morphology workflow to segment the **electrode winding**.

**Why this approach?**
- Very fast and transparent
- Works surprisingly well even on moderate-quality data.
- Useful for (a) quick exploratory analysis on a handful of slices, (b) generating labels for model training datasets.

You’ll likely adjust the thresholds per dataset; start with the defaults below and iterate.

In [ ]:
# Load example slice
im = tiff.imread('data/cell_vol/im100.tif')
plt.imshow(im, cmap = 'gray')
plt.show()

In [ ]:
# Segment the casing (thresholding)
casing = np.where(im > 90, 1, 0)
plt.imshow(casing)
plt.show()

> From the casing segmentation above, analysis of casing eccentricity and denting can be calculated, see notebook #02.

In [ ]:
# Segment the foreground from background 
electrode = np.where(im > 52, 1, 0)
# Removing the casing from the foreground results in a segmentation of the 'electrode'
electrode = np.where(dilation(casing, star(2)) == 1, 0, electrode)
plt.imshow(electrode)
plt.show()

In [ ]:
# The cathode is then isolated from the anode current collector 
# via binary erosion/dilation to remove the thinner anode winding
cathode = dilation(erosion(electrode, star(1)), star(1))
cathode = remove_small_objects(cathode.astype(bool))
plt.imshow(cathode)
plt.show()

> From the electrode winding segmentation above, analysis of the winding quality can be analysed by calculating residuals from a fitted spiral, see notebook #02.

## Conclusion
You now have two complementary routes to segmentation:
- Mini U-Net: compact, accurate on overhangs; can be extended by training on diverse cells and cell types. A similar approach can be followed for other key compenents in the battery (electrode winding, tabs etc.). In the overhang example presented here, downstream analytics can be calculated as in notebook #01.
- Classic CV: immediate and explainable; ideal for quick checks and generating labels for model training. For the example presented here, downstream analytics can be calculated as in notebook #02 for the casing analysis, and notebook #03 for the electrode winding. 